# Steph Curry 2020–2021 NBA Season Performance Analysis
**Author:** Nathan Merrill  
**Institution:** Mountainland Technology  
**Dataset:** `data/stephcurry2020-2021.xlsx` — 28 games, December 2020 – February 2021

---

### Problem Statement
Analyze Steph Curry's 2020–2021 NBA season performance to identify what factors drive his scoring output and how his individual performance relates to Golden State Warriors win/loss outcomes.

### Business Questions
1. How does Curry's scoring and shooting efficiency vary by month?
2. Does playing at home vs. away affect his performance and the team's win rate?
3. Which shooting categories most strongly correlate with winning?
4. How do minutes played relate to points scored and team success?
5. What does his plus/minus reveal about his impact on game outcomes?
6. How do his assists and turnovers relate to wins and losses?


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

BLUE   = '#1F4E79'
MIDBLUE= '#2E75B6'
LTBLUE = '#9DC3E6'
RED    = '#C00000'
GRAY   = '#595959'

## 2. Load Data

In [ ]:
df = pd.read_excel('data/stephcurry2020-2021.xlsx')
print(f"Shape: {df.shape}")
df.head()

## 3. Data Cleaning

**Steps taken:**
- Convert `MIN` from timedelta to numeric minutes
- Verify no missing values
- Check for out-of-range values
- Confirm `RESULT` and `COURT` encoding


In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Convert MIN (stored as timedelta) to integer minutes
df['MIN_num'] = df['MIN'].dt.seconds // 60

print(f"Minutes range: {df['MIN_num'].min()} – {df['MIN_num'].max()} minutes")
print(f"RESULT values: {df['RESULT'].unique()}  (1=Win, 0=Loss)")
print(f"COURT values:  {df['COURT'].unique()}   (H=Home, A=Away)")
print(f"MONTH values:  {df['MONTH'].unique()}")

In [ ]:
# Descriptive statistics for numeric columns
df.describe().round(2)

In [ ]:
# Confirm no data entry errors in key stats
assert df['FG%'].between(0, 100).all(), "FG% out of range"
assert df['3P%'].between(0, 100).all(), "3P% out of range"
assert df['PTS'].between(0, 100).all(), "PTS out of range"
assert df['MIN_num'].between(20, 50).all(), "MIN out of expected range"
print("All range checks passed.")

## 4. Exploratory Data Analysis

In [ ]:
wins   = df['RESULT'].sum()
losses = len(df) - wins

print(f"Games played : {len(df)}")
print(f"Wins         : {wins}")
print(f"Losses       : {losses}")
print(f"Win rate     : {wins/len(df)*100:.1f}%")
print()
print(f"Avg PTS  : {df['PTS'].mean():.1f}")
print(f"Avg 3PM  : {df['3PM'].mean():.1f}")
print(f"Avg FG%  : {df['FG%'].mean():.1f}%")
print(f"Avg 3P%  : {df['3P%'].mean():.1f}%")
print(f"Avg AST  : {df['AST'].mean():.1f}")
print(f"PTS range: {df['PTS'].min()} – {df['PTS'].max()}")

## 5. Analysis & Visualizations

### Q1: How does scoring and efficiency vary by month?
#### Figure 1 — Points per game with rolling average

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

colors = [MIDBLUE if r == 1 else RED for r in df['RESULT']]
bars = ax.bar(range(len(df)), df['PTS'], color=colors, alpha=0.85, zorder=2, width=0.7)

rolling = df['PTS'].rolling(5, center=True).mean()
ax.plot(range(len(df)), rolling, color=BLUE, linewidth=2.5,
        label='5-game rolling avg', zorder=3)
ax.axhline(df['PTS'].mean(), color=GRAY, linewidth=1.3, linestyle='--',
           label=f"Season avg ({df['PTS'].mean():.1f} pts)", zorder=3)

win_patch  = mpatches.Patch(color=MIDBLUE, alpha=0.85, label='Win')
loss_patch = mpatches.Patch(color=RED,     alpha=0.85, label='Loss')
ax.legend(handles=[win_patch, loss_patch] + ax.get_lines(), fontsize=9, loc='upper left')

ax.set_xlabel('Game Number', fontsize=11)
ax.set_ylabel('Points', fontsize=11)
ax.set_title("Steph Curry — Points Per Game (Dec 2020 – Feb 2021)",
             fontsize=13, fontweight='bold', pad=10)
ax.set_xlim(-0.5, 27.5)
ax.set_ylim(0, 70)

plt.tight_layout()
plt.savefig('visualizations/fig1_scoring_trend.png', dpi=150, bbox_inches='tight')
plt.show()

#### Figure 2 — Shooting efficiency by month (FG%, 3P%, FT%)

In [ ]:
month_order = ['DEC', 'JAN', 'FEB']
month_labels = ['December', 'January', 'February']

fg = [df[df.MONTH == m]['FG%'].mean() for m in month_order]
tp = [df[df.MONTH == m]['3P%'].mean() for m in month_order]
ft = [df[df.MONTH == m]['FT%'].mean() for m in month_order]

x = np.arange(3)
w = 0.25
fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - w, fg, w, label='FG%',  color=BLUE,   alpha=0.9)
b2 = ax.bar(x,     tp, w, label='3P%',  color=MIDBLUE, alpha=0.9)
b3 = ax.bar(x + w, ft, w, label='FT%',  color=LTBLUE,  alpha=0.9)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.8,
                f'{bar.get_height():.1f}%',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_labels, fontsize=11)
ax.set_ylabel('Shooting %', fontsize=11)
ax.set_ylim(0, 115)
ax.set_title('Shooting Efficiency by Month — FG%, 3P%, FT%',
             fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('visualizations/fig2_shooting_by_month.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
month_summary = df.groupby('MONTH')[['PTS','FG%','3PM','3P%','RESULT']].mean().round(2)
month_summary.columns = ['Avg PTS','Avg FG%','Avg 3PM','Avg 3P%','Win Rate']
month_summary.loc[:, 'Win Rate'] = (month_summary['Win Rate'] * 100).round(0).astype(str) + '%'
month_summary.index = ['December','January','February'] if set(month_summary.index) == {'DEC','JAN','FEB'} else month_summary.index
print(month_summary)

### Q2: Does home court affect performance and win rate?
#### Figure 3 — Home vs. away comparison

In [ ]:
categories  = ['Avg PTS', 'FG%', '3P%', 'Win Rate %', 'Avg AST']
home_vals   = [
    df[df.COURT=='H']['PTS'].mean(),
    df[df.COURT=='H']['FG%'].mean(),
    df[df.COURT=='H']['3P%'].mean(),
    df[df.COURT=='H']['RESULT'].mean() * 100,
    df[df.COURT=='H']['AST'].mean(),
]
away_vals   = [
    df[df.COURT=='A']['PTS'].mean(),
    df[df.COURT=='A']['FG%'].mean(),
    df[df.COURT=='A']['3P%'].mean(),
    df[df.COURT=='A']['RESULT'].mean() * 100,
    df[df.COURT=='A']['AST'].mean(),
]

x = np.arange(len(categories))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - w/2, home_vals, w, label='Home', color=BLUE,  alpha=0.9)
b2 = ax.bar(x + w/2, away_vals, w, label='Away', color=RED,   alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Home vs. Away — Points, Shooting %, Win Rate, Assists',
             fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('visualizations/fig3_home_vs_away.png', dpi=150, bbox_inches='tight')
plt.show()

print("Home games:", len(df[df.COURT=='H']), "| Away games:", len(df[df.COURT=='A']))
print(df.groupby('COURT')[['PTS','FG%','3P%','RESULT','AST']].mean().round(2))

### Q3: Which stats correlate most with winning?
#### Figure 4 — Pearson correlation of stats with game result

In [ ]:
corr_cols = ['P/M', '3P%', 'TO', '3PM', 'FTM', 'PTS', 'AST', 'FG%', 'MIN_num']
corr_labels = ['Plus/Minus', '3P%', 'Turnovers', '3PM', 'FTM', 'Points', 'Assists', 'FG%', 'Minutes']
corr_vals = [df[c].corr(df['RESULT']) for c in corr_cols]

# Sort descending
pairs = sorted(zip(corr_vals, corr_labels), reverse=True)
vals, labels = zip(*pairs)

bar_colors = [BLUE if v >= 0.2 else MIDBLUE if v >= 0.1 else LTBLUE for v in vals]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(labels[::-1], vals[::-1], color=bar_colors[::-1], alpha=0.9)

for bar, val in zip(bars, vals[::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.axvline(0.2, color=RED, linewidth=1.3, linestyle='--',
           alpha=0.75, label='Moderate threshold (r = 0.20)')
ax.set_xlabel('Pearson Correlation with Win (r)', fontsize=11)
ax.set_title("Correlation of Curry's Stats with Warriors Win",
             fontsize=13, fontweight='bold', pad=10)
ax.set_xlim(0, 0.95)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/fig4_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelation values:")
for label, val in zip(labels, vals):
    print(f"  {label:<15} {val:+.3f}")

### Q4: Do minutes played relate to scoring and team success?

In [ ]:
win_min  = df[df.RESULT==1]['MIN_num'].mean()
loss_min = df[df.RESULT==0]['MIN_num'].mean()
corr_min = df['MIN_num'].corr(df['RESULT'])

print(f"Avg minutes in wins  : {win_min:.1f}")
print(f"Avg minutes in losses: {loss_min:.1f}")
print(f"Correlation (MIN vs WIN): {corr_min:.3f}  — essentially no relationship")
print()

corr_pts = df['MIN_num'].corr(df['PTS'])
print(f"Correlation (MIN vs PTS): {corr_pts:.3f}")

### Q5: What does plus/minus reveal about Curry's impact?
#### Figure 5 — Win vs. loss distributions (box plots)

In [ ]:
win_df  = df[df.RESULT == 1]
loss_df = df[df.RESULT == 0]

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
fig.suptitle("Distribution of Key Stats: Wins vs. Losses",
             fontsize=13, fontweight='bold', y=1.01)

for ax, col, title in zip(axes, ['PTS', '3P%', 'P/M'],
                                  ['Points', '3P%', 'Plus / Minus']):
    bp = ax.boxplot(
        [win_df[col], loss_df[col]],
        patch_artist=True,
        medianprops=dict(color='white', linewidth=2.5),
        whiskerprops=dict(color=GRAY),
        capprops=dict(color=GRAY),
        flierprops=dict(marker='o', markerfacecolor=GRAY, markersize=5, alpha=0.7),
    )
    bp['boxes'][0].set_facecolor(MIDBLUE)
    bp['boxes'][1].set_facecolor(RED)
    ax.set_xticklabels(['Win', 'Loss'], fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('visualizations/fig5_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Avg P/M in wins  : {win_df['P/M'].mean():+.1f}")
print(f"Avg P/M in losses: {loss_df['P/M'].mean():+.1f}")

### Q6: How do assists and turnovers relate to wins and losses?

In [ ]:
ast_to = df.groupby('RESULT')[['AST', 'TO']].mean().round(2)
ast_to.index = ['Loss', 'Win']
ast_to['AST:TO Ratio'] = (ast_to['AST'] / ast_to['TO']).round(2)
print(ast_to)

print()
print(f"Correlation (AST vs WIN): {df['AST'].corr(df['RESULT']):+.3f}")
print(f"Correlation (TO  vs WIN): {df['TO'].corr(df['RESULT']):+.3f}")
print()
print("Note: More turnovers in wins suggests a more aggressive, attack-oriented style.")

## 6. Summary — Win vs. Loss Averages

In [ ]:
summary_cols = ['PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%',
                'FTM', 'AST', 'TO', 'P/M', 'MIN_num']

summary = df.groupby('RESULT')[summary_cols].mean().round(2)
summary.index = ['Loss', 'Win']
summary

## 7. Conclusion

| Finding | Key Stat |
|---|---|
| Best month | February (36.3 PPG, 58.3% FG, 47.6% 3P%) |
| Home court impact | 62% win rate at home vs. 42% away |
| Strongest win predictor | Plus/minus (r = 0.81) |
| Best shooting predictor | 3P% (r = 0.27) |
| Surprising result | More turnovers in wins — aggressive style pays off |
| Minutes impact | Zero correlation with winning (r = 0.00) |

**Strategic takeaways:**
1. Maximizing Curry's 3-point opportunities matters more than overall shot volume
2. The supporting cast needs to elevate away from home to convert Curry's output into wins
3. Plus/minus is the most reliable in-game indicator of team success
